# Case-Based Reasoning (CBR) Research

Notebook ini mengimplementasikan sistem **Case-Based Reasoning (CBR)** untuk klasifikasi diagnosis sakit kepala.

---

## Alur Kerja
1. **Import Library** — Memuat semua pustaka yang diperlukan
2. **Konfigurasi Path** — Mendefinisikan path dataset secara lokal
3. **Load Dataset** — Membaca dan mengeksplorasi dataset
4. **Eksplorasi Data (EDA)** — Missing values, statistik deskriptif, distribusi label
5. **Pembangunan Kamus Kode Kasus** — Memetakan kode kasus ↔ gejala
6. **Encoding Fitur** — One-hot encoding kode kasus
7. **Augmentasi Data** — Random Deletion untuk memperbanyak data
8. **Pemodelan SVM (Data Augmented)** — Training + evaluasi SVM
9. **Pemodelan SVM (Data Asli)** — Baseline SVM tanpa augmentasi
10. **Perbandingan SVM** — Sebelum vs sesudah augmentasi
11. **CBR — Representasi Basis Kasus** — TF-IDF dari data augmented
12. **CBR — Fungsi Inti** — Retrieve, Reuse, Predict
13. **CBR — Demo Query** — Contoh penggunaan nyata
14. **CBR — Evaluasi LOO-CV** — Leave-One-Out Cross-Validation
15. **CBR vs SVM** — Perbandingan performa
16. **CBR — Fase Retain** — Menyimpan kasus baru ke basis kasus

---

## Langkah 1: Import Library

Mengimpor semua pustaka Python yang dibutuhkan untuk analisis data, pemodelan, dan visualisasi.

In [ ]:
import os
import random
import itertools
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import vstack

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

print("Semua library berhasil diimpor.")

## Langkah 2: Konfigurasi Path Dataset

Mendefinisikan path ke file dataset secara dinamis berdasarkan lokasi notebook ini.

In [ ]:
# --- Konfigurasi Path ---
# Path direktori dataset relatif terhadap lokasi notebook ini
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
DATASET_DIR  = os.path.join(NOTEBOOK_DIR, '..', 'Dataset')

# Path file-file dataset
PATH_DATASET_UTAMA     = os.path.join(DATASET_DIR, 'dataset.xlsx')
PATH_DATASET_AUGMENTED = os.path.join(DATASET_DIR, 'dataset_augmented.xlsx')

# Tampilkan path untuk verifikasi
print(f"Direktori Dataset : {os.path.abspath(DATASET_DIR)}")
print(f"Dataset Utama     : {os.path.abspath(PATH_DATASET_UTAMA)}")
print(f"Dataset Augmented : {os.path.abspath(PATH_DATASET_AUGMENTED)}")

# Cek keberadaan file
print("\n--- Status File ---")
print(f"  dataset.xlsx          : {'DITEMUKAN ✓' if os.path.exists(PATH_DATASET_UTAMA) else 'TIDAK DITEMUKAN ✗'}")
print(f"  dataset_augmented.xlsx: {'DITEMUKAN ✓' if os.path.exists(PATH_DATASET_AUGMENTED) else 'BELUM ADA (akan dibuat)'}")

## Langkah 3: Load Dataset

Membaca dataset utama dari file Excel. Dataset berisi kasus pasien dengan gejala yang telah dinormalisasi dan kode diagnosis ICD-X.

In [ ]:
# Membaca dataset utama (header ada di baris ke-2, sehingga header=1)
print(f"Membaca file: {PATH_DATASET_UTAMA}")
df = pd.read_excel(PATH_DATASET_UTAMA, header=1)
df.columns = df.columns.str.strip()  # Bersihkan spasi di nama kolom

print(f"\nDataset berhasil dimuat.")
print(f"Jumlah baris   : {len(df)}")
print(f"Jumlah kolom   : {len(df.columns)}")
print(f"Nama kolom     : {list(df.columns)}")

print("\n--- 5 Data Pertama ---")
display(df.head())

## Langkah 4: Eksplorasi Data (EDA)

Memeriksa kualitas dan karakteristik dataset:
- Missing values
- Statistik deskriptif
- Distribusi label diagnosis

In [ ]:
# 4.1 Cek Missing Values
print("=== Missing Values per Kolom ===")
missing_values = df.isnull().sum()
print(missing_values)
print(f"\nTotal missing values: {missing_values.sum()}")

In [ ]:
# 4.2 Statistik Deskriptif (kolom numerik)
print("=== Statistik Deskriptif ===")
display(df.describe())

In [ ]:
# 4.3 Tampilkan kolom kunci (diagnosis, gejala, dan kode kasus)
print("=== Kolom Kunci: icd_x, normalized_semicolon, case_codes ===")
pd.set_option('display.max_columns', None)
display(df[['icd_x', 'normalized_semicolon', 'case_codes']])

In [ ]:
# 4.4 Distribusi label diagnosis (icd_x)
print("=== Distribusi Label Diagnosis (icd_x) ===")
label_counts = df['icd_x'].value_counts()
print(label_counts)

plt.figure(figsize=(6, 4))
label_counts.plot(kind='bar', color=['steelblue', 'coral'])
plt.title('Distribusi Kelas Diagnosis ICD-X')
plt.xlabel('Kode ICD-X')
plt.ylabel('Jumlah Kasus')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Langkah 5: Pembangunan Kamus Kode Kasus ↔ Gejala

Membangun dua kamus yang saling terkait:
1. **`symptom_to_case_codes`** — Dari setiap gejala, kode kasus apa saja yang dikaitkan
2. **`case_code_to_symptoms`** — Dari setiap kode kasus, gejala apa saja yang muncul

Ini adalah inti dari representasi kasus dalam sistem CBR.

In [ ]:
# --- Bangun kamus: Gejala → Kode Kasus ---
symptom_to_case_codes = {}

for _, row in df.iterrows():
    symptoms   = [s.strip() for s in row['normalized_semicolon'].split(';')] if pd.notna(row['normalized_semicolon']) else []
    case_codes = [c.strip() for c in row['case_codes'].split(',')]           if pd.notna(row['case_codes'])           else []

    for symptom in symptoms:
        if symptom not in symptom_to_case_codes:
            symptom_to_case_codes[symptom] = Counter()
        symptom_to_case_codes[symptom].update(case_codes)

# --- Bangun kamus: Kode Kasus → Gejala ---
case_code_to_symptoms = {}

for _, row in df.iterrows():
    symptoms   = [s.strip() for s in row['normalized_semicolon'].split(';')] if pd.notna(row['normalized_semicolon']) else []
    case_codes = [c.strip() for c in row['case_codes'].split(',')]           if pd.notna(row['case_codes'])           else []

    for code in case_codes:
        if code not in case_code_to_symptoms:
            case_code_to_symptoms[code] = Counter()
        case_code_to_symptoms[code].update(symptoms)

print(f"Jumlah gejala unik yang terpetakan    : {len(symptom_to_case_codes)}")
print(f"Jumlah kode kasus unik yang terpetakan: {len(case_code_to_symptoms)}")

In [ ]:
# Tampilkan 5 gejala terkait teratas untuk setiap kode kasus
print("\n=== Kamus: Kode Kasus → 5 Gejala Teratas ===")
for code, symptoms_counter in case_code_to_symptoms.items():
    print(f"\nKode: '{code}'")
    if symptoms_counter:
        for symptom, count in symptoms_counter.most_common(5):
            print(f"  - {symptom} (muncul {count} kali)")
    else:
        print("  - Tidak ada gejala terkait.")

## Langkah 6: Encoding Fitur (One-Hot Encoding untuk Kode Kasus)

Mengubah kolom `case_codes` (yang berisi daftar kode dipisah koma) menjadi matriks biner.
Setiap kolom merepresentasikan satu kode kasus: nilai `1` = kode ada, `0` = kode tidak ada.

In [ ]:
# Kumpulkan semua kode kasus unik dari seluruh dataset
all_case_codes = []
for codes_str in df['case_codes'].dropna():
    all_case_codes.extend([code.strip() for code in codes_str.split(',')])
unique_case_codes = sorted(set(all_case_codes))

print(f"Jumlah kode kasus unik: {len(unique_case_codes)}")
print(f"Kode kasus: {unique_case_codes}")

# Buat matriks biner (one-hot) untuk kode kasus
case_codes_encoded = pd.DataFrame(0, index=df.index, columns=unique_case_codes)

for index, row in df.iterrows():
    if pd.notna(row['case_codes']):
        codes_in_row = [code.strip() for code in row['case_codes'].split(',')]
        for code in codes_in_row:
            if code in case_codes_encoded.columns:
                case_codes_encoded.loc[index, code] = 1

print("\n--- Matriks Encoding Kode Kasus (5 Baris Pertama) ---")
display(case_codes_encoded.head())

print("\n--- Label (icd_x) 5 Baris Pertama ---")
display(df['icd_x'].head())

## Langkah 7: Augmentasi Data

Memperbanyak data latih menggunakan teknik **Random Deletion**: secara acak menghapus sebagian gejala dari tiap kasus dengan probabilitas tertentu.

- Probabilitas penghapusan: **15%** per gejala
- Jumlah augmentasi: **4x** (sehingga total = 5× data asli = 200 baris)
- Minimal 1 gejala selalu dipertahankan per kasus

Hasil disimpan ke `dataset_augmented.xlsx`.

In [ ]:
random.seed(42)  # Untuk reproduktibilitas

def random_symptom_deletion(baris_teks, deletion_prob=0.15):
    """
    Menghapus gejala secara acak dari teks yang dipisah titik koma.
    Minimal 1 gejala selalu dipertahankan.

    Args:
        baris_teks   (str)  : String gejala yang dipisah oleh ';'
        deletion_prob(float): Probabilitas penghapusan tiap gejala (default 15%)

    Returns:
        str: String gejala hasil augmentasi
    """
    teks = str(baris_teks).strip()
    daftar_gejala = [g.strip() for g in teks.split(';') if g.strip()]

    # Jika hanya 1 gejala, kembalikan langsung tanpa perubahan
    if len(daftar_gejala) <= 1:
        return teks

    # Hapus gejala dengan probabilitas `deletion_prob`
    hasil = [g for g in daftar_gejala if random.random() > deletion_prob]

    # Pastikan minimal ada 1 gejala yang tersisa
    if len(hasil) == 0:
        hasil = [random.choice(daftar_gejala)]

    return ";".join(hasil)

In [ ]:
# --- Proses Augmentasi ---
JUMLAH_AUGMENTASI = 4

if 'normalized_semicolon' not in df.columns:
    print("ERROR: Kolom 'normalized_semicolon' tidak ditemukan di dataset!")
else:
    list_df_augment = []

    print(f"Memulai augmentasi Random Deletion sebanyak {JUMLAH_AUGMENTASI} kali...")

    for i in range(JUMLAH_AUGMENTASI):
        df_temp = df.copy()
        df_temp['normalized_semicolon'] = df_temp['normalized_semicolon'].apply(
            lambda x: random_symptom_deletion(x, deletion_prob=0.15)
        )
        df_temp['sumber_data'] = f'RandomDeletion Augment ke-{i + 1}'
        list_df_augment.append(df_temp)
        print(f"  Augmentasi ke-{i + 1} selesai ({len(df_temp)} baris baru)")

    # Tandai data asli dan gabungkan
    df['sumber_data'] = 'Data Asli'
    df_final = pd.concat([df] + list_df_augment, ignore_index=True)

    # Simpan ke file Excel
    df_final.to_excel(PATH_DATASET_AUGMENTED, index=False)

    print("=" * 45)
    print("PROSES SELESAI!")
    print(f"  Jumlah data awal     : {len(df)} baris")
    print(f"  Jumlah data akhir    : {len(df_final)} baris")
    print(f"  File disimpan ke     : {os.path.abspath(PATH_DATASET_AUGMENTED)}")

## Langkah 8: Pemodelan SVM — Data Augmented

Melatih model **Support Vector Machine (SVM)** dengan kernel linear menggunakan:
- **Fitur**: Representasi TF-IDF dari kolom `normalized_semicolon`
- **Label**: Kode diagnosis `icd_x`
- **Data**: Dataset yang telah diaugmentasi (200 baris)

Rasio split train/test: **80% / 20%**

In [ ]:
print("=== Pemodelan SVM untuk icd_x — Data Augmented ===")

# Load dataset augmented
print(f"Memuat dataset augmented dari: {PATH_DATASET_AUGMENTED}")
df_augmented = pd.read_excel(PATH_DATASET_AUGMENTED)

# Siapkan fitur dan label
X        = df_augmented['normalized_semicolon'].fillna('')
y_icd_x  = df_augmented['icd_x']

# Encode label ke format numerik
label_encoder = LabelEncoder()
y_encoded     = label_encoder.fit_transform(y_icd_x)

print(f"Jumlah sampel        : {len(X)}")
print(f"Jumlah kelas icd_x   : {len(label_encoder.classes_)}")
print(f"Kelas                : {list(label_encoder.classes_)}")

# Split data
X_train, X_test, y_train_enc, y_test_enc = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)
print(f"\nJumlah data training : {len(X_train)}")
print(f"Jumlah data testing  : {len(X_test)}")

# Vektorisasi TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf    = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf     = tfidf_vectorizer.transform(X_test)

# Latih model SVM
print("\nMelatih model SVM (kernel=linear)...")
svm_augmented = SVC(kernel='linear', random_state=42)
svm_augmented.fit(X_train_tfidf, y_train_enc)
print("Pelatihan selesai.")

In [ ]:
# --- Evaluasi Model SVM (Data Augmented) ---
y_pred_enc = svm_augmented.predict(X_test_tfidf)

accuracy           = accuracy_score(y_test_enc, y_pred_enc)
f1_weighted        = f1_score(y_test_enc, y_pred_enc, average='weighted', zero_division=0)
precision_weighted = precision_score(y_test_enc, y_pred_enc, average='weighted', zero_division=0)
recall_weighted    = recall_score(y_test_enc, y_pred_enc, average='weighted', zero_division=0)

print("=== Evaluasi Model SVM — Data Augmented ===")
print(f"  Accuracy           : {accuracy:.4f}")
print(f"  Weighted F1-Score  : {f1_weighted:.4f}")
print(f"  Weighted Precision : {precision_weighted:.4f}")
print(f"  Weighted Recall    : {recall_weighted:.4f}")

print("\n--- Classification Report ---")
print(classification_report(
    y_test_enc, y_pred_enc,
    target_names=label_encoder.classes_,
    zero_division=0
))

# Confusion Matrix
print("--- Confusion Matrix ---")
cm = confusion_matrix(y_test_enc, y_pred_enc)
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title('Confusion Matrix — SVM (Data Augmented)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

In [ ]:
# --- Top 10 Fitur Paling Penting (Berdasarkan Koefisien SVM Linear) ---
print("=== Top 10 Fitur TF-IDF Paling Penting (SVM Augmented) ===")
feature_names = tfidf_vectorizer.get_feature_names_out()

if hasattr(svm_augmented, 'coef_'):
    dense_coef = svm_augmented.coef_.toarray()

    # Untuk binary: gunakan langsung; untuk multi-class: rata-rata absolute
    if len(label_encoder.classes_) > 2:
        abs_coefficients = np.mean(np.abs(dense_coef), axis=0)
    else:
        abs_coefficients = np.abs(dense_coef).flatten()

    top_indices = abs_coefficients.argsort()[-10:][::-1]

    for i, idx in enumerate(top_indices):
        print(f"  {i+1:2d}. {feature_names[idx]:<25} (Magnitude: {abs_coefficients[idx]:.4f})")
else:
    print("Koefisien tidak tersedia untuk kernel SVM ini.")

## Langkah 9: Pemodelan SVM — Data Asli (Pembanding)

Melatih model SVM yang sama namun **hanya menggunakan data asli** (40 baris) tanpa augmentasi, sebagai baseline untuk perbandingan performa.

In [ ]:
print("=== Pemodelan SVM untuk icd_x — Data Asli (Baseline) ===")

# Load dataset asli (tanpa augmentasi)
print(f"Memuat dataset asli dari: {PATH_DATASET_UTAMA}")
df_original = pd.read_excel(PATH_DATASET_UTAMA, header=1)
df_original.columns = df_original.columns.str.strip()

# Siapkan fitur dan label
X_orig = df_original['normalized_semicolon'].fillna('')
y_orig = df_original['icd_x']

# Encode label
label_encoder_orig = LabelEncoder()
y_orig_enc         = label_encoder_orig.fit_transform(y_orig)

print(f"Jumlah sampel        : {len(X_orig)}")
print(f"Jumlah kelas icd_x   : {len(label_encoder_orig.classes_)}")
print(f"Kelas                : {list(label_encoder_orig.classes_)}")

# Split data
X_train_o, X_test_o, y_train_o, y_test_o = train_test_split(
    X_orig, y_orig_enc, test_size=0.2, random_state=42
)
print(f"\nJumlah data training : {len(X_train_o)}")
print(f"Jumlah data testing  : {len(X_test_o)}")

# Vektorisasi TF-IDF
tfidf_orig      = TfidfVectorizer(max_features=5000)
X_train_o_tfidf = tfidf_orig.fit_transform(X_train_o)
X_test_o_tfidf  = tfidf_orig.transform(X_test_o)

# Latih model SVM
print("\nMelatih model SVM (kernel=linear)...")
svm_original = SVC(kernel='linear', random_state=42)
svm_original.fit(X_train_o_tfidf, y_train_o)
print("Pelatihan selesai.")

In [ ]:
# --- Evaluasi Model SVM (Data Asli) ---
y_pred_o = svm_original.predict(X_test_o_tfidf)

accuracy_o  = accuracy_score(y_test_o, y_pred_o)
f1_o        = f1_score(y_test_o, y_pred_o, average='weighted', zero_division=0)
precision_o = precision_score(y_test_o, y_pred_o, average='weighted', zero_division=0)
recall_o    = recall_score(y_test_o, y_pred_o, average='weighted', zero_division=0)

print("=== Evaluasi Model SVM — Data Asli ===")
print(f"  Accuracy           : {accuracy_o:.4f}")
print(f"  Weighted F1-Score  : {f1_o:.4f}")
print(f"  Weighted Precision : {precision_o:.4f}")
print(f"  Weighted Recall    : {recall_o:.4f}")

print("\n--- Classification Report ---")
print(classification_report(
    y_test_o, y_pred_o,
    target_names=label_encoder_orig.classes_,
    zero_division=0
))

# Confusion Matrix
print("--- Confusion Matrix ---")
cm_o = confusion_matrix(y_test_o, y_pred_o)
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm_o, annot=True, fmt="d", cmap="Oranges",
    xticklabels=label_encoder_orig.classes_,
    yticklabels=label_encoder_orig.classes_
)
plt.title('Confusion Matrix — SVM (Data Asli/Baseline)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

## Langkah 10: Perbandingan Performa SVM — Sebelum vs Sesudah Augmentasi

Membandingkan hasil evaluasi model SVM pada data asli (baseline) versus data yang telah diaugmentasi.

In [ ]:
# --- Tabel Perbandingan ---
hasil_perbandingan = pd.DataFrame({
    'Metrik'        : ['Accuracy', 'F1-Score (Weighted)', 'Precision (Weighted)', 'Recall (Weighted)'],
    'Data Asli'     : [accuracy_o,  f1_o,         precision_o,       recall_o],
    'Data Augmented': [accuracy,    f1_weighted,  precision_weighted, recall_weighted]
})
hasil_perbandingan['Selisih'] = hasil_perbandingan['Data Augmented'] - hasil_perbandingan['Data Asli']
hasil_perbandingan = hasil_perbandingan.set_index('Metrik')

print("=== Perbandingan Performa Model SVM ===")
print(hasil_perbandingan.to_string(float_format='{:.4f}'.format))

# Visualisasi perbandingan
fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(hasil_perbandingan))
width = 0.35

bars1 = ax.bar([xi - width/2 for xi in x], hasil_perbandingan['Data Asli'],      width, label='Data Asli',      color='coral')
bars2 = ax.bar([xi + width/2 for xi in x], hasil_perbandingan['Data Augmented'], width, label='Data Augmented', color='steelblue')

ax.set_xlabel('Metrik')
ax.set_ylabel('Nilai')
ax.set_title('Perbandingan Performa SVM: Data Asli vs. Data Augmented')
ax.set_xticks(list(x))
ax.set_xticklabels(hasil_perbandingan.index, rotation=15, ha='right')
ax.set_ylim(0, 1.1)
ax.legend()

# Tambahkan nilai di atas setiap bar
for bar in bars1:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---

# Metode CBR (Case-Based Reasoning) dengan Cosine Similarity

Bagian ini mengimplementasikan sistem CBR menggunakan **data hasil augmentasi** sebagai basis kasus, mengikuti siklus **4R**:

```
Query Baru
    │
    ▼
┌──────────────────────────────────────────────────────────┐
│  1. RETRIEVE  — Ambil kasus paling mirip dari basis kasus │
│                 (data augmented, 200 kasus)               │
│                 menggunakan Cosine Similarity             │
│                 (threshold ≥ 0.70)                        │
│                                                          │
│  2. REUSE     — Weighted majority voting dari kasus       │
│                 yang ditemukan                            │
│                                                          │
│  3. REVISE    — Validasi/koreksi solusi jika diperlukan  │
│                                                          │
│  4. RETAIN    — Simpan kasus baru ke basis kasus         │
└──────────────────────────────────────────────────────────┘
    │
    ▼
Solusi Diagnosis
```

| Parameter | Nilai |
|-----------|-------|
| **Basis kasus** | `dataset_augmented.xlsx` (200 kasus) |
| Representasi kasus | TF-IDF (unigram + bigram) dari `normalized_semicolon` |
| Fungsi kemiripan | **Cosine Similarity** |
| Threshold | **≥ 0.70** |
| Strategi voting | Weighted majority vote (bobot = similarity) |
| Evaluasi | Leave-One-Out Cross-Validation (LOO-CV) |

## Langkah 11: Representasi Basis Kasus dari Data Augmented

Memuat **dataset hasil augmentasi** (200 baris) sebagai basis kasus CBR, kemudian membangun matriks TF-IDF yang menjadi representasi seluruh kasus.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 11.1  Muat dataset AUGMENTED sebagai basis kasus CBR
# ─────────────────────────────────────────────────────────────
print(f"Memuat dataset augmented untuk basis kasus CBR: {PATH_DATASET_AUGMENTED}")
df_cbr = pd.read_excel(PATH_DATASET_AUGMENTED)
df_cbr.columns = df_cbr.columns.str.strip()

# Pastikan tidak ada missing value di kolom kunci
df_cbr = df_cbr.dropna(subset=['normalized_semicolon', 'icd_x']).reset_index(drop=True)

print(f"\nJumlah kasus dalam basis kasus (augmented) : {len(df_cbr)}")
print(f"Distribusi label icd_x:")
print(df_cbr['icd_x'].value_counts().to_string())

# Tampilkan komposisi sumber data (asli vs augmentasi)
if 'sumber_data' in df_cbr.columns:
    print(f"\nKomposisi sumber data:")
    print(df_cbr['sumber_data'].value_counts().to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────
# 11.2  Bangun TF-IDF vectorizer dari seluruh basis kasus augmented
#        Menggunakan unigram + bigram untuk representasi lebih kaya
# ─────────────────────────────────────────────────────────────
THRESHOLD_SIMILARITY = 0.70   # Threshold cosine similarity CBR

cbr_vectorizer   = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
case_base_matrix = cbr_vectorizer.fit_transform(df_cbr['normalized_semicolon'])

print(f"Dimensi matriks TF-IDF basis kasus augmented : {case_base_matrix.shape}")
print(f"  Baris  = jumlah kasus                      : {case_base_matrix.shape[0]}")
print(f"  Kolom  = jumlah fitur (n-gram)             : {case_base_matrix.shape[1]}")
print(f"\nThreshold Cosine Similarity                  : {THRESHOLD_SIMILARITY}")

## Langkah 12: Fungsi Inti CBR

Mendefinisikan fungsi-fungsi utama siklus CBR:
- **`cbr_retrieve()`** — Fase Retrieve: cari kasus terdekat ≥ threshold dari basis kasus augmented
- **`cbr_reuse()`** — Fase Reuse: tentukan solusi via weighted majority voting
- **`cbr_predict()`** — Gabungan Retrieve + Reuse untuk satu query

In [ ]:
def cbr_retrieve(query_text, case_base_tfidf, df_cases,
                 vectorizer, threshold=0.70, top_k=5):
    """
    FASE RETRIEVE: Ambil kasus-kasus paling mirip dari basis kasus augmented.

    Algoritma:
      1. Transformasikan teks query ke vektor TF-IDF
      2. Hitung cosine similarity antara query dengan semua kasus
      3. Filter kasus dengan similarity >= threshold (0.70)
      4. Urutkan dari yang paling mirip, ambil top_k teratas

    Args:
        query_text      (str)            : Teks gejala query baru (dipisah ';')
        case_base_tfidf (sparse matrix)  : Matriks TF-IDF basis kasus
        df_cases        (DataFrame)      : DataFrame basis kasus
        vectorizer      (TfidfVectorizer): Vectorizer yang sudah di-fit
        threshold       (float)          : Minimal cosine similarity (default 0.70)
        top_k           (int)            : Maksimum kasus yang dikembalikan

    Returns:
        DataFrame: Kasus yang memenuhi threshold + kolom 'similarity'.
                   Kosong jika tidak ada kasus yang memenuhi threshold.
    """
    query_vec    = vectorizer.transform([query_text])
    similarities = cosine_similarity(query_vec, case_base_tfidf).flatten()

    df_result = df_cases.copy()
    df_result['similarity'] = similarities
    df_result = df_result[df_result['similarity'] >= threshold]
    df_result = df_result.sort_values('similarity', ascending=False).head(top_k)
    return df_result.reset_index(drop=True)


def cbr_reuse(retrieved_cases):
    """
    FASE REUSE: Tentukan solusi dari kasus-kasus yang ditemukan.

    Strategi: Weighted Majority Voting
      - Tiap kasus memberikan suara untuk label icd_x-nya
      - Bobot suara = nilai cosine similarity
      - Label dengan total bobot tertinggi dipilih sebagai prediksi
      - Confidence = bobot label terpilih / total bobot semua kasus

    Args:
        retrieved_cases (DataFrame): Output dari cbr_retrieve()

    Returns:
        tuple: (predicted_label, confidence_score)
               predicted_label = None jika tidak ada kasus mirip.
    """
    if retrieved_cases.empty:
        return None, 0.0

    vote_scores     = retrieved_cases.groupby('icd_x')['similarity'].sum()
    predicted_label = vote_scores.idxmax()
    total_sim       = retrieved_cases['similarity'].sum()
    confidence      = vote_scores[predicted_label] / total_sim if total_sim > 0 else 0.0

    return predicted_label, round(float(confidence), 4)


def cbr_predict(query_text, case_base_tfidf, df_cases,
                vectorizer, threshold=0.70, top_k=5):
    """
    Gabungan Retrieve + Reuse untuk satu query.

    Returns:
        dict: {
            'prediksi'    : label icd_x atau None,
            'confidence'  : bobot keyakinan (0.0–1.0),
            'kasus_mirip' : DataFrame kasus yang ditemukan,
            'ada_kasus'   : bool
        }
    """
    kasus_mirip    = cbr_retrieve(query_text, case_base_tfidf, df_cases,
                                  vectorizer, threshold, top_k)
    prediksi, conf = cbr_reuse(kasus_mirip)
    return {
        'prediksi'   : prediksi,
        'confidence' : conf,
        'kasus_mirip': kasus_mirip,
        'ada_kasus'  : not kasus_mirip.empty
    }


print("Fungsi CBR (retrieve, reuse, predict) berhasil didefinisikan.")
print(f"Threshold cosine similarity: {THRESHOLD_SIMILARITY}")
print(f"Basis kasus                : {len(df_cbr)} kasus (data augmented)")

## Langkah 13: Demo — Query Kasus Baru

Menjalankan CBR pada satu contoh query gejala baru. Sistem akan mencari kasus mirip dari **basis kasus augmented** (200 kasus), kemudian memberikan prediksi diagnosis.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Contoh query: masukkan gejala baru dipisah titik koma
# ─────────────────────────────────────────────────────────────
QUERY_BARU = "demam 3 hari;pusing;bab lembek;perut mual nihil;muntah nihil;alergi obat nihil"

print("=" * 60)
print("         SISTEM CBR — DEMO QUERY KASUS BARU")
print("=" * 60)
print(f"\n[INPUT]  Gejala Query  : {QUERY_BARU}")
print(f"[CONFIG] Threshold     : {THRESHOLD_SIMILARITY}")
print(f"[CONFIG] Basis Kasus   : {len(df_cbr)} kasus (data augmented)")
print(f"[CONFIG] Max Top-K     : 5")

hasil = cbr_predict(
    query_text      = QUERY_BARU,
    case_base_tfidf = case_base_matrix,
    df_cases        = df_cbr,
    vectorizer      = cbr_vectorizer,
    threshold       = THRESHOLD_SIMILARITY,
    top_k           = 5
)

# --- Fase RETRIEVE ---
print("\n─── FASE RETRIEVE ──────────────────────────────────────")
if hasil['ada_kasus']:
    cols_show = ['icd_x', 'normalized_semicolon', 'similarity']
    if 'sumber_data' in df_cbr.columns:
        cols_show = ['icd_x', 'sumber_data', 'normalized_semicolon', 'similarity']
    kasus_tampil = hasil['kasus_mirip'][cols_show].copy()
    kasus_tampil['normalized_semicolon'] = kasus_tampil['normalized_semicolon'].str[:60] + '...'
    kasus_tampil['similarity'] = kasus_tampil['similarity'].round(4)
    print(f"Ditemukan {len(hasil['kasus_mirip'])} kasus mirip (similarity >= {THRESHOLD_SIMILARITY}):")
    display(kasus_tampil)
else:
    print(f"Tidak ada kasus dengan similarity >= {THRESHOLD_SIMILARITY}.")
    print("Sistem tidak dapat memberikan rekomendasi — perlu konsultasi dokter.")

# --- Fase REUSE ---
print("\n─── FASE REUSE ─────────────────────────────────────────")
if hasil['prediksi']:
    print(f"[OUTPUT] Prediksi Diagnosis  : {hasil['prediksi']}")
    print(f"[OUTPUT] Confidence          : {hasil['confidence']:.2%}")
    print("\nKeterangan kode ICD-X:")
    print("  G44.0 = Cluster headache (Sakit kepala klaster)")
    print("  G44.2 = Tension-type headache (Sakit kepala tegang)")
else:
    print("[OUTPUT] Prediksi : Tidak dapat ditentukan (di bawah threshold)")

print("=" * 60)

## Langkah 14: Evaluasi CBR — Leave-One-Out Cross-Validation (LOO-CV)

Mengevaluasi performa CBR menggunakan **Leave-One-Out Cross-Validation** pada **data augmented** (200 kasus):

1. Setiap kasus (dari 200 kasus augmented) diuji satu per satu sebagai query
2. Kasus yang sedang diuji **dikeluarkan sementara** dari basis kasus (mencegah data leakage)
3. Sisa 199 kasus menjadi basis kasus untuk proses Retrieve
4. Hasilnya dicatat: **BENAR** / **SALAH** / **TIDAK_TERJAWAB**

> ⚠️ **Catatan**: Basis kasus yang lebih besar (200 vs 40) meningkatkan kemungkinan menemukan kasus mirip karena data augmented mencakup variasi gejala yang lebih beragam.

In [ ]:
print("=== Evaluasi CBR — Leave-One-Out Cross-Validation ===")
print(f"Dataset         : {len(df_cbr)} kasus (data augmented)")
print(f"Threshold       : {THRESHOLD_SIMILARITY}")
print("Memulai LOO-CV (200 iterasi)...\n")

y_true_cbr   = []
y_pred_cbr   = []
detail_hasil = []

for i in range(len(df_cbr)):
    # Pisahkan kasus ke-i sebagai query, sisanya sebagai basis kasus
    df_train   = df_cbr.drop(index=i).reset_index(drop=True)
    query_text = df_cbr.loc[i, 'normalized_semicolon']
    true_label = df_cbr.loc[i, 'icd_x']

    # Bangun matriks TF-IDF dari basis kasus sementara (tanpa kasus ke-i)
    train_matrix = cbr_vectorizer.transform(df_train['normalized_semicolon'])

    # Jalankan CBR
    hasil_i = cbr_predict(
        query_text      = query_text,
        case_base_tfidf = train_matrix,
        df_cases        = df_train,
        vectorizer      = cbr_vectorizer,
        threshold       = THRESHOLD_SIMILARITY,
        top_k           = 5
    )

    pred_label  = hasil_i['prediksi']
    confidence  = hasil_i['confidence']
    n_retrieved = len(hasil_i['kasus_mirip'])

    y_true_cbr.append(true_label)
    y_pred_cbr.append(pred_label if pred_label is not None else 'TIDAK_TERJAWAB')

    detail_hasil.append({
        'no'           : i + 1,
        'sumber_data'  : df_cbr.loc[i, 'sumber_data'] if 'sumber_data' in df_cbr.columns else '-',
        'query_singkat': str(query_text)[:55] + '...',
        'label_asli'   : true_label,
        'prediksi_cbr' : pred_label,
        'confidence'   : confidence,
        'n_retrieved'  : n_retrieved,
        'status'       : 'BENAR' if pred_label == true_label else
                         ('TIDAK_TERJAWAB' if pred_label is None else 'SALAH')
    })

    # Progress log setiap 50 iterasi
    if (i + 1) % 50 == 0:
        print(f"  Progress: {i + 1}/{len(df_cbr)} kasus selesai...")

print(f"\nLOO-CV selesai. {len(df_cbr)} kasus dievaluasi.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Tampilkan tabel detail hasil LOO-CV
# ─────────────────────────────────────────────────────────────
df_detail = pd.DataFrame(detail_hasil)

status_counts = df_detail['status'].value_counts()
print("=== Rekap Status Prediksi CBR (LOO-CV, Data Augmented) ===")
for status, cnt in status_counts.items():
    print(f"  {status:<20}: {cnt} kasus")

# Breakdown per sumber data (asli vs augmentasi)
if 'sumber_data' in df_detail.columns:
    print("\n--- Breakdown per Sumber Data ---")
    breakdown = df_detail.groupby(['sumber_data', 'status']).size().unstack(fill_value=0)
    display(breakdown)

print("\n--- Tabel Detail Hasil LOO-CV (10 baris pertama) ---")
display(df_detail[['no', 'sumber_data', 'label_asli', 'prediksi_cbr',
                   'confidence', 'n_retrieved', 'status']].head(10))

In [ ]:
# ─────────────────────────────────────────────────────────────
# Hitung metrik evaluasi CBR
# ─────────────────────────────────────────────────────────────
df_terjawab       = df_detail[df_detail['prediksi_cbr'].notna()]
df_tidak_terjawab = df_detail[df_detail['prediksi_cbr'].isna()]

n_total          = len(df_cbr)
n_terjawab       = len(df_terjawab)
n_tidak_terjawab = len(df_tidak_terjawab)
n_benar          = (df_detail['status'] == 'BENAR').sum()
n_salah          = (df_detail['status'] == 'SALAH').sum()

coverage         = n_terjawab / n_total
accuracy_overall = n_benar / n_total
accuracy_cond    = n_benar / n_terjawab if n_terjawab > 0 else 0.0

y_true_t = df_terjawab['label_asli'].tolist()
y_pred_t = df_terjawab['prediksi_cbr'].tolist()

f1_cbr        = f1_score(y_true_t, y_pred_t, average='weighted', zero_division=0)        if n_terjawab > 0 else 0.0
precision_cbr = precision_score(y_true_t, y_pred_t, average='weighted', zero_division=0) if n_terjawab > 0 else 0.0
recall_cbr    = recall_score(y_true_t, y_pred_t, average='weighted', zero_division=0)    if n_terjawab > 0 else 0.0

avg_confidence  = df_terjawab['confidence'].mean()  if n_terjawab > 0 else 0.0
avg_n_retrieved = df_terjawab['n_retrieved'].mean() if n_terjawab > 0 else 0.0

print("=" * 60)
print("   HASIL EVALUASI CBR — DATA AUGMENTED (LOO-CV)")
print("=" * 60)
print(f"  Basis Kasus                   : {n_total} kasus (augmented)")
print(f"  Threshold Cosine Similarity   : {THRESHOLD_SIMILARITY}")
print(f"  Kasus Terjawab                : {n_terjawab} ({coverage:.1%})")
print(f"  Kasus Tidak Terjawab          : {n_tidak_terjawab} ({1-coverage:.1%})")
print(f"  Prediksi BENAR                : {n_benar}")
print(f"  Prediksi SALAH                : {n_salah}")
print()
print(f"  ── Metrik (semua kasus) ──────────────────────")
print(f"  Accuracy (Overall)            : {accuracy_overall:.4f}")
print(f"  Coverage                      : {coverage:.4f}")
print()
print(f"  ── Metrik (kasus terjawab saja) ──────────────")
print(f"  Accuracy (Conditional)        : {accuracy_cond:.4f}")
print(f"  Weighted F1-Score             : {f1_cbr:.4f}")
print(f"  Weighted Precision            : {precision_cbr:.4f}")
print(f"  Weighted Recall               : {recall_cbr:.4f}")
print(f"  Rata-rata Confidence          : {avg_confidence:.4f}")
print(f"  Rata-rata Kasus Ditemukan     : {avg_n_retrieved:.2f}")
print("=" * 60)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Confusion Matrix CBR (kasus terjawab)
# ─────────────────────────────────────────────────────────────
if n_terjawab > 0:
    classes_cbr = sorted(df_cbr['icd_x'].unique())
    cm_cbr = confusion_matrix(y_true_t, y_pred_t, labels=classes_cbr)

    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm_cbr, annot=True, fmt='d', cmap='Greens',
        xticklabels=classes_cbr,
        yticklabels=classes_cbr
    )
    plt.title(f'Confusion Matrix — CBR Cosine Similarity\n(Data Augmented, threshold = {THRESHOLD_SIMILARITY})')
    plt.xlabel('Prediksi')
    plt.ylabel('Label Asli')
    plt.tight_layout()
    plt.show()

    print("\n--- Classification Report CBR (Kasus Terjawab) ---")
    print(classification_report(y_true_t, y_pred_t,
                                labels=classes_cbr, zero_division=0))

In [ ]:
# ─────────────────────────────────────────────────────────────
# Visualisasi: Distribusi max-similarity + Coverage pie chart
# ─────────────────────────────────────────────────────────────
print("Menghitung distribusi max cosine similarity (200 iterasi)...")
max_sims = []
for i in range(len(df_cbr)):
    df_train_i  = df_cbr.drop(index=i).reset_index(drop=True)
    train_mat_i = cbr_vectorizer.transform(df_train_i['normalized_semicolon'])
    query_vec_i = cbr_vectorizer.transform([df_cbr.loc[i, 'normalized_semicolon']])
    sims_i      = cosine_similarity(query_vec_i, train_mat_i).flatten()
    max_sims.append(float(sims_i.max()))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Plot 1: Histogram distribusi max-similarity
axes[0].hist(max_sims, bins=20, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[0].axvline(THRESHOLD_SIMILARITY, color='crimson', linestyle='--',
                linewidth=2, label=f'Threshold = {THRESHOLD_SIMILARITY}')
axes[0].set_title('Distribusi Max Cosine Similarity (LOO-CV, Data Augmented)')
axes[0].set_xlabel('Max Cosine Similarity')
axes[0].set_ylabel('Jumlah Kasus')
axes[0].legend()

# Plot 2: Pie chart coverage
labels_pie = [f'Terjawab\n({n_terjawab})', f'Tidak Terjawab\n({n_tidak_terjawab})']
sizes_pie  = [n_terjawab, n_tidak_terjawab]
colors_pie = ['mediumseagreen', 'lightcoral']
axes[1].pie(sizes_pie, labels=labels_pie, colors=colors_pie,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
axes[1].set_title(f'Coverage CBR — Data Augmented (threshold = {THRESHOLD_SIMILARITY})')

plt.tight_layout()
plt.show()

## Langkah 15: Perbandingan CBR vs SVM

Membandingkan performa **CBR Cosine Similarity (data augmented)** dengan model **SVM** (data asli dan augmented).

In [ ]:
# ─────────────────────────────────────────────────────────────
# Tabel perbandingan CBR vs SVM
# ─────────────────────────────────────────────────────────────
col_cbr_overall = f'CBR Augmented\n(thr={THRESHOLD_SIMILARITY}, Overall)'
col_cbr_cond    = f'CBR Augmented\n(thr={THRESHOLD_SIMILARITY}, Conditional)'

df_compare = pd.DataFrame({
    'Metrik'               : ['Accuracy', 'F1-Score (Weighted)',
                               'Precision (Weighted)', 'Recall (Weighted)'],
    'SVM — Data Asli'      : [accuracy_o,       f1_o,        precision_o,       recall_o],
    'SVM — Data Augmented' : [accuracy,         f1_weighted, precision_weighted, recall_weighted],
    col_cbr_overall        : [accuracy_overall, f1_cbr,      precision_cbr,      recall_cbr],
    col_cbr_cond           : [accuracy_cond,    f1_cbr,      precision_cbr,      recall_cbr],
})
df_compare = df_compare.set_index('Metrik')

print("=== Perbandingan Performa: CBR (Augmented) vs SVM ===")
print(df_compare.to_string(float_format='{:.4f}'.format))
print(f"\n* CBR Coverage (kasus yang berhasil dijawab): {coverage:.4f} ({coverage:.1%})")
print(f"* Basis kasus CBR: {n_total} kasus (data augmented)")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Visualisasi perbandingan CBR vs SVM
# ─────────────────────────────────────────────────────────────
metrik_labels = ['Accuracy', 'F1-Score\n(Weighted)',
                 'Precision\n(Weighted)', 'Recall\n(Weighted)']

vals_svm_asli = [accuracy_o,       f1_o,        precision_o,       recall_o]
vals_svm_aug  = [accuracy,          f1_weighted, precision_weighted, recall_weighted]
vals_cbr_aug  = [accuracy_overall,  f1_cbr,      precision_cbr,      recall_cbr]

x     = np.arange(len(metrik_labels))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))

b1 = ax.bar(x - width, vals_svm_asli, width, label='SVM — Data Asli',                         color='coral')
b2 = ax.bar(x,          vals_svm_aug,  width, label='SVM — Data Augmented',                    color='steelblue')
b3 = ax.bar(x + width,  vals_cbr_aug,  width, label=f'CBR Augmented (thr={THRESHOLD_SIMILARITY})', color='mediumseagreen')

ax.set_xlabel('Metrik')
ax.set_ylabel('Nilai')
ax.set_title('Perbandingan Performa: CBR Cosine Similarity (Data Augmented) vs SVM')
ax.set_xticks(x)
ax.set_xticklabels(metrik_labels)
ax.set_ylim(0, 1.20)
ax.legend(loc='lower right')

# Anotasi nilai di atas bar
for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.3f}',
                    xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', fontsize=8.5)

plt.tight_layout()
plt.show()

## Langkah 16: Fase RETAIN — Menyimpan Kasus Baru ke Basis Kasus

Fase **RETAIN** adalah fase terakhir siklus CBR. Setelah solusi divalidasi oleh dokter, kasus baru ditambahkan ke **basis kasus augmented** agar sistem terus berkembang.

In [ ]:
def cbr_retain(df_case_base, new_case_text, true_label, vectorizer,
               sumber='Kasus Baru'):
    """
    FASE RETAIN: Tambahkan kasus baru ke basis kasus dan
    perbarui representasi TF-IDF.

    Args:
        df_case_base  (DataFrame)      : Basis kasus saat ini
        new_case_text (str)            : Teks gejala kasus baru (dipisah ';')
        true_label    (str)            : Label diagnosis yang benar/tervalidasi
        vectorizer    (TfidfVectorizer): Vectorizer yang sudah di-fit
        sumber        (str)            : Keterangan sumber data

    Returns:
        tuple: (df_updated DataFrame, matrix_updated sparse matrix)
    """
    new_row = pd.DataFrame([{
        'normalized_semicolon': new_case_text,
        'icd_x'              : true_label,
        'sumber_data'        : sumber
    }])
    df_updated     = pd.concat([df_case_base, new_row], ignore_index=True)
    matrix_updated = vectorizer.transform(df_updated['normalized_semicolon'])
    return df_updated, matrix_updated


# ─── Contoh penggunaan RETAIN ───────────────────────────────
KASUS_BARU_TEXT = "kepala pusing berdenyut sebelah kanan;mual;tidak nafsu makan"
LABEL_VALIDASI  = "G44.0"   # Dikonfirmasi oleh dokter

print("=== FASE RETAIN — Menyimpan Kasus Baru ke Basis Kasus ===")
print(f"  Kasus baru           : {KASUS_BARU_TEXT}")
print(f"  Label (tervalidasi)  : {LABEL_VALIDASI}")
print(f"  Basis kasus sebelum  : {len(df_cbr)} kasus (augmented)")

df_cbr_updated, case_base_matrix_updated = cbr_retain(
    df_case_base  = df_cbr,
    new_case_text = KASUS_BARU_TEXT,
    true_label    = LABEL_VALIDASI,
    vectorizer    = cbr_vectorizer,
    sumber        = 'Kasus Baru Tervalidasi'
)

print(f"  Basis kasus sesudah  : {len(df_cbr_updated)} kasus")
print("\n✓ Kasus baru berhasil ditambahkan ke basis kasus augmented!")

# --- Verifikasi ---
print("\n─── Verifikasi: Query Mirip dengan Kasus yang Baru Ditambahkan ─")
query_verif = "kepala berdenyut;mual;tidak mau makan"
verifikasi  = cbr_predict(
    query_text      = query_verif,
    case_base_tfidf = case_base_matrix_updated,
    df_cases        = df_cbr_updated,
    vectorizer      = cbr_vectorizer,
    threshold       = THRESHOLD_SIMILARITY
)
print(f"  Query          : {query_verif}")
print(f"  Prediksi       : {verifikasi['prediksi']}")
print(f"  Confidence     : {verifikasi['confidence']:.2%}")
print(f"  Kasus ditemukan: {len(verifikasi['kasus_mirip'])} kasus")